# Экспорт `trial.params` в JSON для `--params-json`

Формат — плоский объект, как у завершённого trial transformer-поиска (те же ключи, что в Optuna `FrozenTrial.params`). Его читает **`scripts/run_transformer_fixed_params_all_patients.py`**.

Путь к SQLite совпадает с тем, что даёт `run_optuna_transformer_all_patients.py` (`out-dir / tfr_<id>_transformer.db`). Имя study обычно равно stem файла.


In [ ]:
import json
import sys
from pathlib import Path
from typing import cast

from optuna.trial import TrialState

_cwd = Path.cwd().resolve()
_cands = [_cwd, _cwd.parent, *_cwd.parents[:3]]
project_root = next((p for p in _cands if (p / "lib" / "optuna").is_dir()), None)
if project_root is None:
    raise FileNotFoundError(
        "NeuronDeCo root not found (expected lib/optuna). cwd=" + str(_cwd)
    )
sys.path.insert(0, str(project_root))

from lib.optuna import load_study_sqlite

# --- правьте под свой запуск ---
STUDY_DB = Path("../../PreprocessedData/2026-05-11/tfr_s11_transformer.db")
STUDY_NAME = STUDY_DB.stem  # или строка явно, если отличается

# Если None — берём правило «top-5 по F1 → минимум второй цели», как в свечном ноутбуке.
# cast(...): иначе статический анализатор сужает тип до Literal[None] и помечает ветку
# `if TRIAL_NUMBER is not None` как unreachable.
TRIAL_NUMBER = cast(int | None, None)

OUT_JSON = Path("../../PreprocessedData/2026-05-11/tfr_s11_transformer_best_params.json")


def select_best_trial_top5_f1_then_min_loss(study):
    complete_trials = [
        t
        for t in study.get_trials(deepcopy=False)
        if t.state == TrialState.COMPLETE and t.values is not None and len(t.values) >= 2
    ]
    if not complete_trials:
        return None
    ranked_by_f1 = sorted(complete_trials, key=lambda t: float(t.values[0]), reverse=True)
    top5 = ranked_by_f1[:5]
    return min(top5, key=lambda t: float(t.values[1]))


study_db = STUDY_DB.expanduser().resolve()
study = load_study_sqlite(db_path=study_db, study_name=STUDY_NAME)

if TRIAL_NUMBER is not None:
    trial = next(
        (t for t in study.get_trials(deepcopy=False) if int(t.number) == TRIAL_NUMBER),
        None,
    )
    if trial is None:
        raise RuntimeError(f"No trial #{TRIAL_NUMBER}")
else:
    trial = select_best_trial_top5_f1_then_min_loss(study)
    if trial is None:
        raise RuntimeError("No suitable COMPLETE trial for auto-pick")

flat = dict(trial.params)
OUT_JSON = OUT_JSON.expanduser().resolve()
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(flat, f, indent=2, ensure_ascii=False)

meta_path = OUT_JSON.with_suffix(".meta.json")
meta = {
    "study_db": str(study_db),
    "study_name": STUDY_NAME,
    "trial_number": int(trial.number),
    "trial_values": list(map(float, trial.values)) if trial.values else None,
    "params_json": str(OUT_JSON),
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("Wrote:", OUT_JSON)
print("Meta:", meta_path)
print("trial", trial.number, "values", trial.values)
print("keys:", sorted(flat.keys()))